# Data Preprocessing

In [1]:
import os
import math
import json
import librosa

DATASET_PATH = "../genres"
JSON_PATH = "../data/processed/data.json"

SAMPLE_RATE = 22050
TRACK_DURATION = 30
SAMPLES_PER_TRACK = SAMPLE_RATE * TRACK_DURATION

## Preprocessing Function

In [2]:
def save_mfcc(dataset_path, json_path, num_mfcc=13, n_fft=2048, hop_length=512, num_segments=10):
    data = {
        "mapping": [],
        "labels": [],
        "mfcc": []
    }

    samples_per_segment = int(SAMPLES_PER_TRACK / num_segments)
    num_mfcc_vectors_per_segment = math.ceil(samples_per_segment / hop_length)

    print("Starting Feature Extraction...")
    
    for i, (dirpath, dirnames, filenames) in enumerate(os.walk(dataset_path)):

        if dirpath is not dataset_path:

            semantic_label = dirpath.split(os.path.sep)[-1]
            data["mapping"].append(semantic_label)
            print(f"\nProcessing Genre: {semantic_label}")

            for f in filenames:

                if not f.endswith('.au'):
                    continue

                file_path = os.path.join(dirpath, f)
                try:
                    signal, sample_rate = librosa.load(file_path, sr=SAMPLE_RATE)
                except Exception as e:
                    print(f"Error loading {file_path}: {e}")
                    continue

                for d in range(num_segments):

                    start = samples_per_segment * d
                    finish = start + samples_per_segment

                    mfcc = librosa.feature.mfcc(y=signal[start:finish], 
                                                  sr=sample_rate, 
                                                  n_mfcc=num_mfcc, 
                                                  n_fft=n_fft, 
                                                  hop_length=hop_length)
                    mfcc = mfcc.T

                    if len(mfcc) == num_mfcc_vectors_per_segment:
                        data["mfcc"].append(mfcc.tolist())
                        data["labels"].append(i-1)

    print(f"\nFinished! Saving extracted features to {json_path} ...")
    with open(json_path, "w") as fp:
        json.dump(data, fp, indent=4)
    print("Saved successfully!")

## Run the Function

In [3]:
save_mfcc(DATASET_PATH, JSON_PATH, num_segments=10)

Starting Feature Extraction...

Processing Genre: blues

Processing Genre: classical

Processing Genre: country

Processing Genre: disco

Processing Genre: hiphop

Processing Genre: jazz

Processing Genre: metal

Processing Genre: pop

Processing Genre: reggae

Processing Genre: rock

Finished! Saving extracted features to ../data/processed/data.json ...
Saved successfully!
